# Llama

This notebook uses an off the shelf Llama model to classify a news article into one of five bias labels. We freeze Llama and add a classification head to create a baseline Llama implementation before going on to LoRa in the next notebook. 

# Import

In [1]:
import os
import pandas as pd
import numpy as np

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, make_scorer, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# if using Google Colab, mount Google Drive
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    train_path = '/content/drive/MyDrive/CS5242Project/transformers_train_preprocessed.csv'
    test_path  = '/content/drive/MyDrive/CS5242Project/transformers_test_preprocessed.csv'
else:
    train_path = 'transformers_train_preprocessed.csv'
    test_path  = 'transformers_test_preprocessed.csv'

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

# TODO remove later
train['cleaned_text'] = train['Text']
test['cleaned_text'] = train['Text']

train['cleaned_text'] = train['cleaned_text'].astype(str)
test['cleaned_text'] = test['cleaned_text'].astype(str)

from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train['Bias'])
y_test  = label_encoder.transform(test['Bias'])
num_classes = len(label_encoder.classes_)

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)

class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        encodings = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": encodings["input_ids"].squeeze(0),
            "attention_mask": encodings["attention_mask"].squeeze(0),
            "labels": torch.tensor(label)
        }

Using device: cuda


In [2]:
torch.manual_seed(42)
np.random.seed(42)

In [3]:
train_dataset = NewsDataset(train['cleaned_text'], y_train, tokenizer)
test_dataset = NewsDataset(test['cleaned_text'], y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8)

In [4]:
# Load TinyLlama and wrap with classification head

class LlamaClassifier(nn.Module):
    def __init__(self, model_id, num_classes):
        super().__init__()
        self.llama = AutoModel.from_pretrained(model_id)
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)

        for param in self.llama.parameters():
            param.requires_grad = False  # Optional: freeze llama

        self.classifier = nn.Linear(self.llama.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask=None):
        outputs = self.llama(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state  # (batch_size, seq_len, hidden_size)

        pooled = hidden_states.mean(dim=1)  # Mean pooling, same as bert
        logits = self.classifier(pooled)
        return logits

In [5]:
model = LlamaClassifier(model_id, num_classes)
model.to(device)


optimizer = AdamW(model.parameters(), lr=0.01, weight_decay=0.01)
num_epochs = 20
total_steps = len(train_loader) * num_epochs
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(y_train), y=y_train)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

In [6]:
# count number of trainable parameters
trainable_params = 0

all_params = 0
for name, param in model.named_parameters():
    all_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()

print(f"Trainable parameters: {trainable_params:,} ({100 * trainable_params / all_params:.2f}% of all parameters)")

Trainable parameters: 10,245 (0.00% of all parameters)


We choose the smallest Llama model because our compute power is limited by our local laptop. 

In [7]:
import time
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

bad_epochs = [0] * num_epochs

patience = 3
best_score = None
model_name = 'llama_base'
fpath = f'{model_name}.pt'
if 'google.colab' in sys.modules:
    fpath = os.path.join('/content/drive/My Drive', fpath)

min_delta = 0.0

for epoch in range(num_epochs):
    start_time = time.time()

    #  Training
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        # Move batch to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        logits = model(input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)

        # Backward pass
        loss.backward()
        optimizer.step()
        scheduler.step()

        optimizer.zero_grad()

        total_train_loss += loss.item()
    avg_train_loss = total_train_loss / len(train_loader)
    print(f"Epoch {epoch+1} →  training loss: {avg_train_loss:.4f}")

    model.eval()
    total_valid_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in train_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            logits = model(input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)

            total_valid_loss += loss.item()

            preds = torch.argmax(logits, dim=-1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    avg_valid_loss = total_valid_loss / len(train_loader)
    valid_acc = accuracy_score(all_labels, all_preds)
    valid_f1 = f1_score(all_labels, all_preds, average='macro')
    current_lr = optimizer.param_groups[0]['lr']
    duration = time.time() - start_time

    print(f"Epoch {epoch+1:2d} | "
          f"train_loss: {avg_train_loss:.4f} | "
          f"valid_acc: {valid_acc:.4f} | "
          f"valid_f1: {valid_f1:.4f} | "
          f"valid_loss: {avg_valid_loss:.4f} | "
          f"lr: {current_lr:.6f} | "
          f"dur: {duration:.2f}s")
    # early stopping
    score = valid_acc
    if best_score is None or (score - best_score) > min_delta:
        best_score = score
        torch.save(model.state_dict(), fpath)
    else:
        bad_epochs[epoch] = 1
        if epoch - 1 > patience and all(bad_epochs[epoch-patience: epoch]):
            print(f"No improvement for {patience} epochs, stopping early")
            break

state_dict = torch.load(fpath)
os.remove(fpath)
model.load_state_dict(state_dict)    


Epoch 1 →  training loss: 1.7561
Epoch  1 | train_loss: 1.7561 | valid_acc: 0.5887 | valid_f1: 0.5413 | valid_loss: 1.3369 | lr: 0.005000 | dur: 143.62s
Epoch 2 →  training loss: 2.4698
Epoch  2 | train_loss: 2.4698 | valid_acc: 0.6505 | valid_f1: 0.5136 | valid_loss: 3.2374 | lr: 0.010000 | dur: 143.77s
Epoch 3 →  training loss: 3.2854
Epoch  3 | train_loss: 3.2854 | valid_acc: 0.7248 | valid_f1: 0.6648 | valid_loss: 1.7213 | lr: 0.009444 | dur: 143.86s
Epoch 4 →  training loss: 2.0361
Epoch  4 | train_loss: 2.0361 | valid_acc: 0.6642 | valid_f1: 0.6019 | valid_loss: 2.0862 | lr: 0.008889 | dur: 143.93s
Epoch 5 →  training loss: 1.5403
Epoch  5 | train_loss: 1.5403 | valid_acc: 0.7666 | valid_f1: 0.7110 | valid_loss: 1.3072 | lr: 0.008333 | dur: 144.14s
Epoch 6 →  training loss: 1.4456
Epoch  6 | train_loss: 1.4456 | valid_acc: 0.8227 | valid_f1: 0.7500 | valid_loss: 1.3488 | lr: 0.007778 | dur: 143.96s
Epoch 7 →  training loss: 0.8554
Epoch  7 | train_loss: 0.8554 | valid_acc: 0.8164

<All keys matched successfully>

## Evaluation

In [8]:
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
model.eval()
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids     = batch["input_ids"].to(device)
        attention_mask= batch["attention_mask"].to(device)
        labels        = batch["labels"].to(device)

        logits = model(input_ids, attention_mask=attention_mask)
        preds  = torch.argmax(logits, dim=-1)
        probs  = torch.softmax(logits, dim=-1)

        all_labels.extend(labels.cpu().numpy())    # flat list of true labels
        all_preds.extend(preds.cpu().numpy())     # flat list of preds
        all_probs.append(probs.cpu().numpy())     # list of (batch_size, n_classes)


In [9]:
y_true  = np.array(all_labels)
y_pred  = np.array(all_preds)
y_score = np.concatenate(all_probs, axis=0)

In [10]:
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

acc = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='weighted')
recall = recall_score(all_labels, all_preds, average='weighted')
f1 = f1_score(all_labels, all_preds, average='weighted')
print(f"Epoch {epoch+1} — Eval accuracy: {acc:.4f}; F1: {f1:.4f}; Precision: {precision:.4f}; Recall: {recall:.4f}")


Epoch 20 — Eval accuracy: 0.3295; F1: 0.3328; Precision: 0.3364; Recall: 0.3295


In [11]:
from sklearn.metrics import classification_report
print(classification_report(all_labels, all_preds, target_names=label_encoder.classes_, digits=4))

              precision    recall  f1-score   support

      center     0.0588    0.0732    0.0652        41
   lean left     0.1864    0.1803    0.1833        61
  lean right     0.0588    0.0541    0.0563        37
        left     0.5324    0.5180    0.5251       222
       right     0.1688    0.1711    0.1699        76

    accuracy                         0.3295       437
   macro avg     0.2011    0.1993    0.2000       437
weighted avg     0.3364    0.3295    0.3328       437



In [12]:
import matplotlib.pyplot as plt
import seaborn as sns
#labels = label_encoder.classes_
label_order = ['left', 'lean left', 'center', 'lean right', 'right']
label_indices = [np.where(label_encoder.classes_ == label)[0][0] for label in label_order]

cm = confusion_matrix(all_labels, all_preds, labels=label_indices)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_order, yticklabels=label_order)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('LoRA Llama Classifier Confusion Matrix')
plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt
from itertools import cycle

classes = label_encoder.classes_
y_test_binarized = label_binarize(y_true, classes=np.arange(len(classes)))

fpr, tpr, roc_auc = {}, {}, {}
for i, c in enumerate(classes):
    fpr[c], tpr[c], _ = roc_curve(y_test_binarized[:, i], y_score[:, i])
    roc_auc[c]        = roc_auc_score(y_test_binarized[:, i], y_score[:, i])

# Plot
plt.figure(figsize=(10, 8))
colors = cycle(['aqua', 'darkorange', 'cornflowerblue', 'red', 'green'])
for c, color in zip(classes, colors):
    plt.plot(fpr[c], tpr[c], color=color, lw=2,
             label=f'{c} (AUC = {roc_auc[c]:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlim(0, 1); plt.ylim(0, 1.05)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('LoRA Llama One-vs-Rest ROC Curves')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()